# 04.2 · Modelagem — redução de dimensionalidade

**CRISP-DM fase 4 — modelagem** · Telco Customer Churn

---

Entrada: a base pré-processada do notebook [`03-preparacao`](03-preparacao.ipynb).

Este notebook não treina agrupamentos. Ele aplica duas reduções de dimensionalidade e mostra **quais colunas mais pesam** em cada uma.

| Redução | Tipo | Como medir a relevância de cada coluna |
|---|---|---|
| **PCA** | linear | **cargas** (*loadings*): o peso de cada coluna em cada componente, ponderado pela variância que o componente explica |
| **UMAP** | não linear | não tem cargas. Um **modelo substituto** (random forest) aprende a prever as coordenadas do UMAP a partir das colunas, e a importância de cada coluna nesse modelo é a relevância |

Como o one-hot espalha uma variável em várias colunas, a relevância também é somada por **variável de origem** (ex.: `zoneamento_RL` + `zoneamento_RM` → `zoneamento`).

In [ ]:
import warnings
warnings.filterwarnings('ignore')  # silencia avisos do numba/umap

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor  # modelo substituto que explica o UMAP
from umap import UMAP                               # pacote umap-learn

VARIANCIA_ALVO = 0.90  # fração da variância que o PCA deve manter
TOP = 15               # quantas colunas aparecem nos rankings

## 1 · Dados

In [ ]:
base = pd.read_csv('../data/processed/Telcon-Customer-Churn_preprocessed.csv', sep=',')  # saída do 03: features prontas para o modelo
#base = base.drop(columns=['preco_venda', 'subclasse'], errors='ignore')            # preço fica para avaliar; subclasse é código, não medida
colunas = base.columns                                                             # nomes das features (one-hot como variavel_categoria)
X = base.to_numpy(dtype=float)

df = pd.read_csv('../data/processed/Telcon-Customer-Churn_tratado.csv', sep=',')  # base curada: preço para colorir e nomes das variáveis originais
variavel = pd.Series({c: max((o for o in df.columns if c == o or c.startswith(f'{o}_')), key=len, default=c)
                      for c in colunas})                                 # cada feature → variável de origem (desfaz o one-hot)

print(f'{X.shape[0]} Clientes × {X.shape[1]} features, vindas de {variavel.nunique()} variáveis')

## 2 · PCA

O PCA encontra as direções de maior variância. Cada componente principal (PC) é uma combinação linear das colunas, e a **carga** diz quanto cada coluna entra nela, com sinal. Ficam os componentes que somam 90% da variância.

In [ ]:
pca_completo = PCA(svd_solver='full').fit(X)                         # todos os componentes, para a curva de variância
acumulada = np.cumsum(pca_completo.explained_variance_ratio_)         # variância explicada acumulada
n = int(np.searchsorted(acumulada, VARIANCIA_ALVO) + 1)               # menor nº de componentes que atinge o alvo

pca = PCA(n_components=n, svd_solver='full').fit(X)                  # PCA final, só com os componentes mantidos
X_pca = pca.transform(X)                                              # base reduzida

plt.figure(figsize=(9, 3.5))
plt.plot(np.arange(1, len(acumulada) + 1), acumulada, '-')
plt.axhline(VARIANCIA_ALVO, color='gray', ls=':')                     # alvo de variância
plt.axvline(n, color='tab:red', ls='--', label=f'{n} componentes')    # onde o alvo é atingido
plt.title(f'PCA · {X.shape[1]} features → {n} componentes ({acumulada[n - 1]:.1%} da variância)')
plt.xlabel('nº de componentes')
plt.ylabel('variância acumulada')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
cargas = pd.DataFrame(pca.components_.T, index=colunas,
                      columns=[f'PC{i + 1}' for i in range(n)])        # carga de cada coluna em cada componente
peso_pc = pca.explained_variance_ratio_ / pca.explained_variance_ratio_.sum()  # importância de cada componente

contrib_pca = (cargas ** 2 * peso_pc).sum(axis=1).sort_values(ascending=False)  # contribuição de cada coluna (soma 1)
contrib_pca_var = contrib_pca.groupby(variavel).sum().sort_values(ascending=False)  # somada por variável de origem

# As colunas que mais pesam nos 5 primeiros componentes, com o sinal da carga
pd.DataFrame({
    f'{pc} ({pca.explained_variance_ratio_[i]:.1%})': [f'{c} ({v:+.2f})' for c, v in
        cargas[pc].reindex(cargas[pc].abs().sort_values(ascending=False).index[:8]).items()]
    for i, pc in enumerate(cargas.columns[:5])
}, index=range(1, 9))

In [ ]:
topo = contrib_pca.index[:TOP]                                       # colunas de maior contribuição
limite = cargas.loc[topo].iloc[:, :5].abs().max().max()               # escala simétrica para as cores
cargas.loc[topo].iloc[:, :5].style.background_gradient(cmap='RdBu_r', vmin=-limite, vmax=limite).format('{:+.2f}')  # azul = carga negativa, vermelho = positiva

In [ ]:
def barras(serie, titulo, ax, cor):
    serie.head(TOP)[::-1].plot.barh(ax=ax, color=cor)                 # maiores no topo
    ax.set_title(titulo)
    ax.grid(alpha=0.3, axis='x')


fig, (ax_col, ax_var) = plt.subplots(1, 2, figsize=(14, 6))
barras(contrib_pca, 'PCA · contribuição por coluna', ax_col, 'tab:blue')
barras(contrib_pca_var, 'PCA · contribuição por variável de origem', ax_var, 'tab:blue')
fig.tight_layout()
plt.show()

## 3 · UMAP

O UMAP preserva a vizinhança entre imóveis, mas não é linear e por isso não tem cargas. Ele roda com 5 componentes, que é o tamanho usado para agrupar, e com `min_dist=0`, que deixa os grupos compactos. Uma versão em 2D serve só para o gráfico.

In [ ]:
umap = UMAP(n_components=5, n_neighbors=30, min_dist=0.0, random_state=42)  # n_neighbors: quanto da vizinhança cada cliente enxerga
X_umap = umap.fit_transform(X)                                              # base reduzida, para agrupar
componentes_umap = [f'UMAP{i + 1}' for i in range(X_umap.shape[1])]

X_umap_2d = UMAP(n_components=2, n_neighbors=30, min_dist=0.1, random_state=42).fit_transform(X)  # só para visualizar

preco = np.log(df['MonthlyCharges'])  # TotalCharges em log, só para colorir: não entrou na redução
fig, (ax_pca, ax_umap) = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, Y, titulo in ((ax_pca, X_pca, 'PCA · PC1 × PC2'), (ax_umap, X_umap_2d, 'UMAP · 2D')):
    pontos = ax.scatter(Y[:, 0], Y[:, 1], c=preco, s=6, cmap='viridis', alpha=0.7)
    ax.set_title(titulo)
fig.colorbar(pontos, ax=[ax_pca, ax_umap], label='log(preço de venda)')
plt.show()

In [ ]:
substituto = RandomForestRegressor(n_estimators=300, min_samples_leaf=5, oob_score=True,
                                   n_jobs=-1, random_state=42).fit(X, X_umap)  # aprende a prever as coordenadas do UMAP
print(f'R² fora da amostra (OOB) do substituto: {substituto.oob_score_:.3f}')    # quão bem as colunas explicam o UMAP

imp_umap = pd.Series(substituto.feature_importances_, index=colunas).sort_values(ascending=False)  # relevância por coluna (soma 1)
imp_umap_var = imp_umap.groupby(variavel).sum().sort_values(ascending=False)                      # somada por variável de origem

fig, (ax_col, ax_var) = plt.subplots(1, 2, figsize=(14, 6))
barras(imp_umap, 'UMAP · importância por coluna', ax_col, 'tab:orange')
barras(imp_umap_var, 'UMAP · importância por variável de origem', ax_var, 'tab:orange')
fig.tight_layout()
plt.show()

In [ ]:
# Direção: correlação de Spearman entre cada coluna relevante e cada componente do UMAP
correlacao = (
    pd.concat([base, pd.DataFrame(X_umap, columns=componentes_umap)], axis=1)
    .corr(method='spearman')
    .loc[imp_umap.index[:TOP], componentes_umap]
)
correlacao.style.background_gradient(cmap='RdBu_r', vmin=-1, vmax=1).format('{:+.2f}')  # azul = cai com o componente, vermelho = sobe

## 4 · PCA × UMAP

In [ ]:
def ranking(serie):
    topo = serie.head(TOP)
    return pd.Series([f'{v} ({c:.1%})' for v, c in topo.items()], index=range(1, len(topo) + 1))  # posição → variável (peso)


comparativo = pd.DataFrame({'PCA': ranking(contrib_pca_var), 'UMAP': ranking(imp_umap_var)})  # lado a lado

em_ambos = sorted(set(contrib_pca_var.index[:TOP]) & set(imp_umap_var.index[:TOP]))  # relevantes nas duas reduções
print(f'No top {TOP} das duas reduções ({len(em_ambos)}): {em_ambos}')
comparativo

In [ ]:
pd.DataFrame(X_pca, columns=cargas.columns).to_csv('../data/processed/Telcon-Customer-Churn_pca.csv', sep=',', index=False)      # base reduzida pelo PCA
pd.DataFrame(X_umap, columns=componentes_umap).to_csv('../data/processed/Telcon-Customer-Churn_umap.csv', sep=',', index=False)  # base reduzida pelo UMAP
print(f'Salvas: PCA {X_pca.shape} · UMAP {X_umap.shape}')

## Como ler o que saiu

* **Relevância na redução não é relevância para o negócio.** Uma coluna pesa muito no PCA porque **varia muito**, e não porque separa segmentos. Com o MinMax, uma coluna binária dividida meio a meio tem variância 0,25. Uma contínua concentrada perto de zero tem bem menos. Por isso o PCA tende a favorecer one-hot equilibrados e indicadores `tem_*`.
* **Sinais das cargas:** colunas com o mesmo sinal num componente andam juntas. Sinais opostos descrevem um eixo, por exemplo "novo e grande" × "antigo e pequeno".
* **O UMAP é explicado por aproximação.** A importância vem do modelo substituto, e só vale se o R² OOB for alto. A importância por impureza da random forest também favorece colunas contínuas em relação às binárias. A soma por variável de origem atenua esse viés.
* **Variáveis no top das duas reduções** são as candidatas mais firmes a descrever a estrutura da carteira. Se o top do UMAP for dominado por poucas categorias raras, ele está isolando nichos, não segmentando.
* **Próximo passo:** as bases `Telcon-Customer-Churn_pca.csv` e `Telcon-Customer-Churn_umap.csv` podem entrar na varredura do 04-0 no lugar da base do 03. A silhueta deve continuar sendo medida no espaço original, para que a comparação seja justa.